# FitPredict AI: dataset inspection

This notebook inspects the selected CSV and creates exploratory charts. It does not train or evaluate a model. The separate training and holdout evaluation commands are documented in the project README.

The default file is `data/processed/calories.csv`. Confirm its source and license before redistributing it; that information was not supplied with the project files.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = Path.cwd() / 'data' / 'processed' / 'calories.csv'
# Prefer the existing DATA_PATH, but fall back to likely project-relative locations.
if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
else:
    candidate_paths = [
        Path.cwd() / 'data' / 'processed' / 'calories.csv',
        Path.cwd().parent / 'data' / 'processed' / 'calories.csv',
        Path.cwd() / 'datasets' / 'calories.csv',
        Path.cwd().parent / 'datasets' / 'calories.csv',
        Path('c:/IACES/data/processed/calories.csv'),
    ]
    for candidate in candidate_paths:
        if candidate.exists():
            DATA_PATH = candidate
            df = pd.read_csv(candidate)
            break
    else:
        raise FileNotFoundError(
            f"Could not find calories.csv. Checked: {DATA_PATH} and "
            f"{[str(p) for p in candidate_paths]}"
        )
print(f'File: {DATA_PATH}')
print(f'Rows: {len(df):,} | Columns: {len(df.columns)}')
display(df.head())

In [ ]:
# Schema, missingness, duplicates, and descriptive summary.
display(pd.DataFrame({'dtype': df.dtypes.astype(str), 'missing': df.isna().sum(), 'missing_pct': (df.isna().mean() * 100).round(2)}))
print('Duplicate rows:', int(df.duplicated().sum()))
display(df.describe(include='all').T)

In [ ]:
# The application uses these five predictors and the Calories target.
selected_columns = ['Gender', 'Age', 'Weight', 'Duration', 'Heart_Rate', 'Calories']
missing_columns = sorted(set(selected_columns) - set(df.columns))
if missing_columns:
    print('Columns to resolve before using this CSV:', missing_columns)
else:
    selected = df[selected_columns].copy()
    display(selected.groupby('Gender', dropna=False)['Calories'].agg(['count', 'mean', 'median', 'std']))
    display(selected.select_dtypes(include='number').corr(numeric_only=True).round(2))

In [ ]:
if not missing_columns:
    numeric = selected.select_dtypes(include='number')
    numeric.hist(figsize=(12, 8), bins=30, edgecolor='white')
    plt.tight_layout()
    plt.show()

In [ ]:
if not missing_columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(data=selected, x='Calories', bins=30, kde=True, ax=axes[0])
    axes[0].set_title('Calorie target distribution')
    sns.boxplot(data=selected, x='Gender', y='Calories', ax=axes[1])
    axes[1].set_title('Calories by gender')
    plt.tight_layout()
    plt.show()

## Notes to record after inspection

- Record the dataset publisher, source URL, and license after verifying them.
- Check whether calories are measured, simulated, or formula-derived. Do not describe synthetic or derived values as measured.
- Review the target distribution and potential outliers before interpreting model metrics.
- The training script makes a reproducible holdout split and compares four regressors. Run it only when you are ready to train.